# المهمة الثانية: تجهيز البيانات

يقرأ هذا الدفتر بيانات الطلبات وبيانات الوجبات ومراكز التجهيز، ثم يدمجها ويفحص جودتها دون تعديل البيانات الخام.

**ملاحظة المسار:** طُلب استخدام `data/raw/`، لكن هذا المجلد غير موجود في المشروع. المسار الموجود للملفات هو `data/dataraw/`، لذلك يستخدمه الدفتر كما هو ولا ينشئ مجلدًا بديلًا.

تقتصر مكتبات الشيفرة على `pandas`. لا تُحذف أو تُعالج القيم الشاذة تلقائيًا؛ الغرض هو رصدها وتوثيقها.

## 1. قراءة ملفات البيانات الخام

تُقرأ الملفات من `data/dataraw/` فقط. يطبع الدفتر عدد الصفوف ومجموع الطلبات في بيانات التدريب قبل البدء بعمليات الدمج.

In [14]:
import pandas as pd


# نحدد مجلد البيانات الموجود فعليًا في المشروع.
# نحدد مجلد البيانات نسبةً إلى مجلد الدفتر.
data_dir = "../data/dataraw/"

# نقرأ ملفات البيانات من المجلد الصحيح.
train = pd.read_csv(data_dir + "train.csv")
meal_info = pd.read_csv(data_dir + "meal_info.csv")
center_info = pd.read_csv(data_dir + "fulfilment_center_info.csv")

# نتحقق من أعمدة الربط والطلب المطلوبة قبل التحليل.
required_train = {"meal_id", "center_id", "num_orders", "checkout_price", "base_price"}
assert required_train.issubset(train.columns), "أعمدة مطلوبة مفقودة من train.csv"
assert "meal_id" in meal_info.columns, "عمود meal_id مفقود من meal_info.csv"
assert "center_id" in center_info.columns, "عمود center_id مفقود من fulfilment_center_info.csv"

# نطبع حجم بيانات التدريب ومجموع الطلبات قبل الدمج الأول.
print(f"قبل أي دمج: الصفوف = {len(train):,} | مجموع num_orders = {train['num_orders'].sum():,.0f}")

قبل أي دمج: الصفوف = 456,548 | مجموع num_orders = 119,557,485


## 2. دمج بيانات الوجبات

يُستخدم دمج يساري للحفاظ على جميع صفوف الطلبات. يتحقق `validate="m:1"` من عدم تكرار مفتاح الوجبة في جدول الوجبات، ويكشف مؤشر الدمج المفاتيح التي لم تجد تطابقًا.

In [15]:
# نثبت عدد الصفوف ومجموع الطلبات مباشرة قبل دمج الوجبات.
rows_before_meal = len(train)
orders_before_meal = train["num_orders"].sum()
print(f"قبل دمج meal_info: الصفوف = {rows_before_meal:,} | مجموع num_orders = {orders_before_meal:,.0f}")

# نربط بيانات الوجبات مع الطلبات دون إسقاط صفوف الطلبات.
meal_merge = train.merge(
    meal_info,
    on="meal_id",
    how="left",
    validate="m:1",
    indicator="_meal_merge",
)

# نطبع عدد الصفوف ومجموع الطلبات فور انتهاء الدمج.
print(f"بعد دمج meal_info: الصفوف = {len(meal_merge):,} | مجموع num_orders = {meal_merge['num_orders'].sum():,.0f}")

# نتحقق من سلامة عدد الصفوف ومجموع الطلبات بعد الدمج.
assert len(meal_merge) == rows_before_meal, "تغير عدد الصفوف بعد دمج الوجبات"
assert meal_merge["num_orders"].sum() == orders_before_meal, "تغير مجموع الطلبات بعد دمج الوجبات"

# نعرض عدد مفاتيح الوجبات المتطابقة وغير المتطابقة.
print("حالة مطابقة meal_id:")
print(meal_merge["_meal_merge"].value_counts(dropna=False))

# نزيل عمود التشخيص فقط بعد تسجيل نتيجة المطابقة.
meal_merge = meal_merge.drop(columns="_meal_merge")

قبل دمج meal_info: الصفوف = 456,548 | مجموع num_orders = 119,557,485
بعد دمج meal_info: الصفوف = 456,548 | مجموع num_orders = 119,557,485
حالة مطابقة meal_id:
_meal_merge
both          456548
left_only          0
right_only         0
Name: count, dtype: int64


## 3. دمج بيانات مراكز التجهيز

نكرر التحقق نفسه عند الربط عبر `center_id`. لا تُحذف الطلبات التي لا تجد مركزًا مطابقًا؛ تُسجل كحالات غير متطابقة لتسهيل المراجعة.

In [16]:
# نطبع عدد الصفوف ومجموع الطلبات قبل دمج بيانات المراكز.
rows_before_center = len(meal_merge)
orders_before_center = meal_merge["num_orders"].sum()
print(f"قبل دمج fulfilment_center_info: الصفوف = {rows_before_center:,} | مجموع num_orders = {orders_before_center:,.0f}")

# نربط بيانات المراكز مع نتائج دمج الوجبات بدمج يساري.
prepared_data = meal_merge.merge(
    center_info,
    on="center_id",
    how="left",
    validate="m:1",
    indicator="_center_merge",
)

# نطبع الإحصاءات مباشرة بعد دمج المراكز.
print(f"بعد دمج fulfilment_center_info: الصفوف = {len(prepared_data):,} | مجموع num_orders = {prepared_data['num_orders'].sum():,.0f}")

# نؤكد أن الدمج لم يكرر الصفوف أو يغير الطلبات.
assert len(prepared_data) == rows_before_center, "تغير عدد الصفوف بعد دمج المراكز"
assert prepared_data["num_orders"].sum() == orders_before_center, "تغير مجموع الطلبات بعد دمج المراكز"

# نعرض عدد مفاتيح المراكز المتطابقة وغير المتطابقة.
print("حالة مطابقة center_id:")
print(prepared_data["_center_merge"].value_counts(dropna=False))

# نزيل عمود التشخيص بعد فحص نتيجة الدمج.
prepared_data = prepared_data.drop(columns="_center_merge")

# نعرض عينة للتحقق من شكل البيانات المجهزة.
prepared_data.head()

قبل دمج fulfilment_center_info: الصفوف = 456,548 | مجموع num_orders = 119,557,485
بعد دمج fulfilment_center_info: الصفوف = 456,548 | مجموع num_orders = 119,557,485
حالة مطابقة center_id:
_center_merge
both          456548
left_only          0
right_only         0
Name: count, dtype: int64


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,city_code,region_code,center_type,op_area
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,647,56,TYPE_C,2.0
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,647,56,TYPE_C,2.0
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,647,56,TYPE_C,2.0
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,647,56,TYPE_C,2.0
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,647,56,TYPE_C,2.0


## 4. القيم المفقودة وأنواع الأعمدة

يعرض الجدول عدد القيم المفقودة ونوع كل عمود في ملفات المصدر والبيانات بعد الدمج. لا تُجرى تعبئة أو إزالة للقيم.

In [17]:
# نجمع عدد القيم المفقودة في كل عمود ولكل إطار بيانات.
missing_summary = pd.concat(
    {
        "train": train.isna().sum(),
        "meal_info": meal_info.isna().sum(),
        "center_info": center_info.isna().sum(),
        "prepared_data": prepared_data.isna().sum(),
    },
    axis=1,
).fillna(0).astype("int64")

# نعرض الأعمدة التي تحتوي على قيم مفقودة فقط.
missing_summary.loc[missing_summary.sum(axis=1) > 0]

,train,meal_info,center_info,prepared_data


In [ ]:
# نعرض أنواع الأعمدة وأعداد القيم غير الفارغة بعد الدمج.
dtype_summary = pd.DataFrame(
    {
        "dtype": prepared_data.dtypes.astype(str),
        "non_null_count": prepared_data.notna().sum(),
        "missing_count": prepared_data.isna().sum(),
    }
)

# نعرض تقرير الأنواع للاطلاع على سلامة القراءة والدمج.
dtype_summary

## 5. فحص التكرار والمفاتيح

يُفحص تكرار الصفوف بالكامل، وتكرار `id` في الطلبات، وتكرار مفاتيح جداول التعريف. هذه الفحوصات لا تحذف أي سجل.

In [18]:
# نحسب الصفوف المكررة بالكامل في ملفات البيانات الثلاثة والنتيجة المدمجة.
duplicate_summary = pd.DataFrame(
    {
        "duplicate_full_rows": {
            "train": int(train.duplicated().sum()),
            "meal_info": int(meal_info.duplicated().sum()),
            "center_info": int(center_info.duplicated().sum()),
            "prepared_data": int(prepared_data.duplicated().sum()),
        },
        "duplicate_key_rows": {
            "train.id": int(train["id"].duplicated().sum()) if "id" in train.columns else None,
            "meal_info.meal_id": int(meal_info["meal_id"].duplicated().sum()),
            "center_info.center_id": int(center_info["center_id"].duplicated().sum()),
        },
    }
)

# نعرض أعداد التكرار دون إزالة السجلات.
duplicate_summary

,duplicate_full_rows,duplicate_key_rows
train,0.0,NaN
meal_info,0.0,NaN
center_info,0.0,NaN
prepared_data,0.0,NaN
train.id,NaN,0.0
meal_info.meal_id,NaN,0.0
center_info.center_id,NaN,0.0


## 6. فحص صلاحية الأسعار والقيم الشاذة

نحدد الأسعار غير الموجبة والحالات التي يتجاوز فيها `checkout_price` قيمة `base_price`. ولرصد القيم المتطرفة، نحسب حدود IQR لكل `meal_id` على حدة كي لا نخلط مستويات أسعار وجبات مختلفة. يُعد هذا تنبيهًا إحصائيًا للمراجعة وليس حكمًا تلقائيًا بأن السعر خطأ.

In [19]:
# نحدد سجلات الأسعار غير الموجبة دون حذفها.
nonpositive_checkout = prepared_data["checkout_price"] <= 0
nonpositive_base = prepared_data["base_price"] <= 0

# نحدد الحالات التي يتجاوز فيها سعر الدفع السعر الأساسي.
checkout_above_base = prepared_data["checkout_price"] > prepared_data["base_price"]

# نلخص عدد الحالات ونسبتها من إجمالي الصفوف.
price_rule_summary = pd.DataFrame(
    {
        "count": [
            int(nonpositive_checkout.sum()),
            int(nonpositive_base.sum()),
            int(checkout_above_base.sum()),
        ],
        "percent_of_rows": [
            nonpositive_checkout.mean() * 100,
            nonpositive_base.mean() * 100,
            checkout_above_base.mean() * 100,
        ],
    },
    index=["checkout_price <= 0", "base_price <= 0", "checkout_price > base_price"],
)

# نعرض ملخص قواعد فحص الأسعار.
price_rule_summary

,count,percent_of_rows
checkout_price <= 0,0,0.000000
base_price <= 0,0,0.000000
checkout_price > base_price,116101,25.430185


In [20]:
# نحسب حدود IQR لسعر الدفع داخل كل وجبة لتجنب مقارنة وجبات متفاوتة السعر.
checkout_q1 = prepared_data.groupby("meal_id")["checkout_price"].transform("quantile", q=0.25)
checkout_q3 = prepared_data.groupby("meal_id")["checkout_price"].transform("quantile", q=0.75)
checkout_iqr = checkout_q3 - checkout_q1
checkout_outlier = (prepared_data["checkout_price"] < checkout_q1 - 1.5 * checkout_iqr) | (
    prepared_data["checkout_price"] > checkout_q3 + 1.5 * checkout_iqr
)

# نحسب حدود IQR للسعر الأساسي داخل كل وجبة.
base_q1 = prepared_data.groupby("meal_id")["base_price"].transform("quantile", q=0.25)
base_q3 = prepared_data.groupby("meal_id")["base_price"].transform("quantile", q=0.75)
base_iqr = base_q3 - base_q1
base_outlier = (prepared_data["base_price"] < base_q1 - 1.5 * base_iqr) | (
    prepared_data["base_price"] > base_q3 + 1.5 * base_iqr
)

# نلخص القيم المتطرفة ونسبة كل نوع من الصفوف.
iqr_summary = pd.DataFrame(
    {
        "count": [int(checkout_outlier.sum()), int(base_outlier.sum())],
        "percent_of_rows": [checkout_outlier.mean() * 100, base_outlier.mean() * 100],
    },
    index=["checkout_price خارج حدود IQR للوجبة", "base_price خارج حدود IQR للوجبة"],
)

# نعرض ملخص القيم المتطرفة حسب حدود IQR الخاصة بكل وجبة.
iqr_summary

,count,percent_of_rows
checkout_price خارج حدود IQR للوجبة,30233,6.622086
base_price خارج حدود IQR للوجبة,38017,8.327054


In [21]:
# نعرض عينة من حالات تجاوز checkout_price للسعر الأساسي للمراجعة.
prepared_data.loc[
    checkout_above_base,
    ["id", "week", "center_id", "meal_id", "checkout_price", "base_price", "num_orders"],
].head(10)

,id,week,center_id,meal_id,checkout_price,base_price,num_orders
1,1466964,1,55,1993,136.83,135.83,270
4,1448490,1,55,2631,243.50,242.50,40
8,1025244,1,55,2707,193.06,192.06,472
12,1446016,1,55,2290,311.43,310.43,162
15,1181556,1,55,2640,282.33,281.33,108
17,1067069,1,55,2126,486.00,485.00,28
18,1058482,1,55,2826,306.58,305.58,188
27,1336534,1,55,1445,628.62,627.62,13
28,1242186,1,55,2444,627.62,626.62,15
29,1012819,1,55,2867,628.62,626.62,13


## 7. خلاصة التحقق

يحافظ الدفتر على جميع صفوف التدريب عبر الدمجين اليساريين، ويتحقق من ذلك بمقارنة عدد الصفوف ومجموع `num_orders` قبل كل دمج وبعده. راجع الجداول أعلاه لمتابعة القيم المفقودة والتكرار والأسعار غير الاعتيادية. لم تُحذف أو تُملأ أي قيمة، ولم تُعدّل ملفات البيانات الخام.